In [1]:
from safetensors.torch import load_file
import torch

In [ ]:
ckpt_weights = torch.load("../pretrained_weights/MASt3R_ViTLarge_BaseDecoder_512_catmlpdpt_metric.pth", map_location='cpu', weights_only=False)
dusterweights = torch.load("../pretrained_weights/MASt3R_ViTLarge_BaseDecoder_512_catmlpdpt_metric.pth", map_location='cpu', weights_only=False)

In [ ]:
model_dict = ckpt_weights['model']
args_dict = ckpt_weights['args']

: 

In [ ]:
print(args_dict)
''' 
AsymmetricMASt3R(enc_depth=24, dec_depth=12, enc_embed_dim=1024, dec_embed_dim=768, enc_num_heads=16, 
                 dec_num_heads=12, pos_embed='RoPE100',img_size=(512, 512), head_type='catmlp+dpt', 
                 output_mode='pts3d+desc24', depth_mode=('exp', -inf, inf), conf_mode=('exp', 1, inf), 
                 patch_embed_cls='PatchEmbedDust3R', two_confs=True, desc_conf_mode=('exp', 0, inf))
'''

In [ ]:
# keys = list(model_dict.keys())

# with open("keys.txt", "w") as f:
#     for key in keys:
#         f.write(key + "\n")

In [ ]:
# print(len(keys))

In [3]:
fast3r_weights = load_file("../pretrained_weights/Fast3R_ViT_Large_512/model.safetensors")


In [8]:
# check the weights that start with decoder.dec_blocks.15
for key in fast3r_weights.keys():
    if key.startswith("decoder.dec_blocks.16"):
        print(key)
        print(fast3r_weights[key].shape)
        print(fast3r_weights[key])
        print("===" * 20)

decoder.dec_blocks.16.attn.proj.bias
torch.Size([1024])
tensor([ 0.0262, -0.0033, -0.0248,  ...,  0.0393, -0.0011, -0.0265])
decoder.dec_blocks.16.attn.proj.weight
torch.Size([1024, 1024])
tensor([[-0.0184,  0.0009,  0.0160,  ...,  0.0354,  0.0078,  0.0043],
        [ 0.0419,  0.0150, -0.0406,  ..., -0.0256,  0.0013, -0.0391],
        [-0.0276,  0.0074, -0.0174,  ..., -0.0117, -0.0424,  0.0464],
        ...,
        [-0.0190, -0.0074, -0.0080,  ...,  0.0172, -0.0492, -0.0006],
        [-0.0023, -0.0025, -0.0235,  ...,  0.0045, -0.0083, -0.0131],
        [-0.0720, -0.0128, -0.0230,  ..., -0.0223,  0.0431, -0.0267]])
decoder.dec_blocks.16.attn.qkv.bias
torch.Size([3072])
tensor([ 0.0696,  0.0111,  0.0069,  ...,  0.0137, -0.0031, -0.0225])
decoder.dec_blocks.16.attn.qkv.weight
torch.Size([3072, 1024])
tensor([[-0.0083,  0.0386, -0.0322,  ...,  0.0368, -0.0062,  0.0253],
        [-0.0079, -0.0038, -0.0184,  ..., -0.0335,  0.0190,  0.0221],
        [-0.0167, -0.0161, -0.0296,  ..., -0.0256,

In [ ]:
fast3r_keys = list(fast3r_weights.keys())
print(len(fast3r_keys))

with open("fast3r_keys.txt", "w") as f:
    for key in fast3r_keys:
        f.write(key + "\n")

In [ ]:
# dusterweights = torch.load("../pretrained_weights/MASt3R_ViTLarge_BaseDecoder_512_catmlpdpt_metric.pth", map_location='cpu', weights_only=False)

In [ ]:
duster_dict = dusterweights['model']
# duster_dict

In [ ]:
duster_dict['head_local_features.fc1.weight']

In [ ]:
len("downstream_head_local")

In [ ]:
final_out = {}
name ={"downstream_head1.dpt.scratch.layer_rn.0.weight":"downstream_head.dpt.scratch.layer1_rn.weight", 
       "downstream_head1.dpt.scratch.layer_rn.1.weight":"downstream_head.dpt.scratch.layer2_rn.weight",
       "downstream_head1.dpt.scratch.layer_rn.2.weight":"downstream_head.dpt.scratch.layer3_rn.weight",
       "downstream_head1.dpt.scratch.layer_rn.3.weight":"downstream_head.dpt.scratch.layer4_rn.weight",}
name_keys = list(name.keys())

name_2 ={"downstream_head2.dpt.scratch.layer_rn.0.weight":"downstream_head_local.dpt.scratch.layer1_rn.weight", 
       "downstream_head2.dpt.scratch.layer_rn.1.weight":"downstream_head_local.dpt.scratch.layer2_rn.weight",
       "downstream_head2.dpt.scratch.layer_rn.2.weight":"downstream_head_local.dpt.scratch.layer3_rn.weight",
       "downstream_head2.dpt.scratch.layer_rn.3.weight":"downstream_head_local.dpt.scratch.layer4_rn.weight",}
name_keys_2 = list(name_2.keys())

with open("hybrid_keys.txt", "r") as f:
    for key in f:
        key = key.strip()
        if 'encoder.' in key : 
            final_out[key[8:]] = fast3r_weights[key]
        # elif 'decoder.dec_norm' in key:
        #     final_out[key[8:]] = model_dict[key[8:]]
        elif key in name_keys:
            final_out[key] = fast3r_weights[name[key]]
        elif key in name_keys_2:
            final_out[key] = fast3r_weights[name_2[key]]
        elif 'decoder.' in key:
            final_out[key[8:]] = fast3r_weights[key]
        elif 'downstream_head_local' in key:
            final_out["downstream_head2"+key[21:]] = fast3r_weights[key]
        elif 'downstream_head' in key:
            final_out["downstream_head1"+key[15:]] = fast3r_weights[key]
        else:
            final_out[key] = model_dict[key]

In [ ]:
final_out_keys = list(final_out.keys())
print(len(final_out_keys))

with open("final_keys.txt", "w") as f:
    for key in final_out_keys:
        f.write(key + "\n")

In [ ]:
print(len(final_out))

In [ ]:
ckpt_weights['model'] = final_out

In [ ]:
torch.save(ckpt_weights, '../pretrained_weights/Fast3R_MASt3R_ViTLarge_BaseDecoder_512_catmlpdpt_metric.pth')

In [ ]:
def fast3r_checkpoint_filter_fn(fast3r_state_dict, nopo_state_dict):
    """ convert patch embedding weight from manual patchify + linear proj to conv"""
    fast3r_out_dict = {}
    nopo_enc_dict = {}
    # state_dict = state_dict.get('model', state_dict)
    # state_dict = state_dict.get('state_dict', state_dict)

    for k, v in fast3r_state_dict.items():
        if 'encoder.enc_blocks' in k:
            fast3r_out_dict[k] = v
            if k[8:] not in nopo_state_dict:
                nopo_enc_dict[k[8:]] = v

    
    if len(nopo_enc_dict) != 0:
        print("there are different weight names! check by p nopo_enc_dict")
        import pdb; pdb.set_trace()


    for k3, v3 in fast3r_out_dict.items():
        key = k3[8:]
        nopo_state_dict[key] = v3
    # add prefix to make our model happy

    return nopo_state_dict

In [ ]:
nopo_dict = fast3r_checkpoint_filter_fn(fast3r_weights, model_dict)

In [ ]:
# nopo_keys = list(nopo_dict.keys())

# with open("nopo_keys.txt", "w") as f:
#     for key in nopo_keys:
#         f.write(key + "\n")